In [ ]:
!pip install transformers torch gradio -q


In [41]:
import torch
import pandas as pd
import torch.nn.functional as F
import gradio as gr

In [44]:
from google.colab import files

uploaded = files.upload()
df = pd.read_csv(list(uploaded.keys())[0])
print(f"عدد الأسئلة: {len(df)}")
df.head()

Saving hotelsense_data.csv to hotelsense_data (6).csv
عدد الأسئلة: 295


,question,answer,category
0,كام سعر الليلة؟,الأسعار تبدأ من 300 ريال لليلة الواحدة، وبتختل...,السعر والحجز
1,إيه سعر الأوضة؟,الأسعار تبدأ من 300 ريال لليلة الواحدة، وبتختل...,السعر والحجز
2,الأسعار بتبدأ من كام؟,الأسعار تبدأ من 300 ريال لليلة الواحدة، وبتختل...,السعر والحجز
3,عايزة أعرف تكلفة الإقامة,الأسعار تبدأ من 300 ريال لليلة الواحدة، وبتختل...,السعر والحجز
4,فيه خصم لو حجزت أسبوع؟,آه، الإقامة لأكثر من أسبوع بتاخد خصم يصل إلى 1...,السعر والحجز


In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings


In [ ]:
question_embeddings = torch.cat([get_embedding(str(q)) for q in df['question']], dim=0)

In [46]:
def ask(user_question, threshold=0.6):
    q_embedding = get_embedding(user_question)
    scores = F.cosine_similarity(q_embedding, question_embeddings)
    best_idx = scores.argmax().item()
    best_score = scores[best_idx].item()

    if best_score < threshold:
        return "عذرًا، السؤال ده مش موجود في قاعدة بياناتنا حاليًا."
    return df.iloc[best_idx]['answer']

# اختبار الموديل

In [47]:
print(ask("كام سعر الليلة؟"))
print(ask("فيه واي فاي؟"))
print(ask("عندكم قطط؟"))

الأسعار تبدأ من 300 ريال لليلة الواحدة، وبتختلف حسب نوع الأوضة والإطلالة.
آه، واي فاي مجاني متاح في كل الأوض والمناطق العامة.
بعض الفنادق بتوفر خطة دفع على دفعتين، اسألي الاستقبال عند الحجز.


In [48]:
import gradio as gr

gr.Interface(
      fn=ask,
    inputs=gr.Textbox(label="اكتبي سؤالك"),
    outputs=gr.Textbox(label="الإجابة"),
    title="🕋 HotelSense"
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7fdd71fa16c2ac063a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
